In [ ]:
import sys
from pathlib import Path
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# Model saving
import os
import joblib

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

root_path = Path.cwd().parent 
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

from pipelines.data_pipeline import  split_data, get_data_and_process_target, scale_features, label_encode_target
from pipelines.Classification_pipelines import evaluate_classification_model

print("Libraries imported successfully!")

## Load Processed Data

In [ ]:
# Initialize your app variables
TARGET = 'Severity'

# 1. Use the component to load data
df, target_stats = get_data_and_process_target("city_traffic_processed.csv", target_column=TARGET)

# 2. Access your range and std whenever you need them
if target_stats:
    print(f"\nReady to process models for {TARGET}...")

In [ ]:
# Check class balance
class_counts = df[TARGET].value_counts()
class_percentages = df[TARGET].value_counts(normalize=True) * 100

print("Class Distribution:")
for cat in class_counts.index:
    print(f"{cat}: {class_counts[cat]} ({class_percentages[cat]:.1f}%)")

# Check for severe imbalance
min_class_pct = class_percentages.min()
if min_class_pct < 10:
    print(f"\nWarning: Smallest class is only {min_class_pct:.1f}% of data.")
    print("Consider adjusting your binning strategy.")
else:
    print(f"\nClass balance looks reasonable!")

## Prepare Features and Target

In [ ]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET]

In [ ]:
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures used: {X.columns.tolist()}")
print(f"\nTarget classes: {y.unique().tolist()}")

## Label encoding

Used to change severity from 1 - 4 to 0 - 3

In [ ]:
# This fixes the "Expected [0 1 2 3]" error.
y_encoded, le = label_encode_target(df['Severity'])

## Train-Test Split

In [ ]:
#Uses SMOTE oversampling to balance classes in the training set, then splits into train/test sets
X_train, X_test, y_train, y_test = split_data(X, y_encoded, test_size=0.2, random_state=42)

## Class Weights

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)

class_weights = dict(zip(classes, weights))

for k,v in class_weights.items():    
    print(f"{k}: {v:.2f}")

## Feature Scaling

In [ ]:
# Scales features using StandardScaler
X_train_scaled, X_test_scaled, scaler, features = scale_features(X_train, X_test)

## SMOTE 

(Only on the scaled training data) 

In [ ]:
# SMOTE 
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

## HistGradientBoostingClassifier

In [ ]:
# Train
hist_model = HistGradientBoostingClassifier(max_iter=100, max_depth=5, random_state=42)
hist_model.fit(X_train_res, y_train_res)

# Evaluate
hist_results, hist_model, y_test_pred = evaluate_classification_model(
    hist_model, X_train_res, X_test_scaled, y_train_res, y_test, 'Hist Gradient'
)

#Store
all_results.append(hist_results)
trained_models['Gradient Boosting'] = hist_model

# --- 4. View Results ---
import pandas as pd
results_df = pd.DataFrame(all_results)
print(results_df)

In [ ]:

# Dictionary to store trained models
trained_models = {
    'Gradient Boosting': baseline_trained
}

# XGBClassifier

In [ ]:
from xgboost import XGBClassifier

#Remove scale_pos_weight from the model definition
model = XGBClassifier(
    n_estimators=100, 
    max_depth=5, 
    gamma=1,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    n_jobs=-1,
    eval_metric='mlogloss',
    random_state=42,
    objective='multi:softprob'
)

model.fit(X_train_scaled, y_train)

In [ ]:
model.fit(X_train_scaled, y_train)

In [ ]:
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

In [ ]:
print (f"'Train Accuracy': {accuracy_score(y_train, y_train_pred):.4f}")

In [ ]:
print (f"'Test Accuracy': {accuracy_score(y_test, y_test_pred):.4f}")

In [ ]:
print (f"'Precision (weighted)': {precision_score(y_test, y_test_pred, average='weighted'):.4f}")

In [ ]:
print (f"'Recall (weighted)': {recall_score(y_test, y_test_pred, average='weighted'):.4f}")

In [ ]:
print (f"'F1 (weighted)': {f1_score(y_test, y_test_pred, average='weighted'):.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_test_pred, target_names=["1", "2", "3", "4"]))
print(confusion_matrix(y_test, y_test_pred))

### Model 1: Traffic Accident Severity — Traditional ML

- Multi-class classification: predict accident severity (1-4 scale)
- This dataset has significant class imbalance — your approach to handling it will be a key evaluation criterion. **Weighted F1 is the real evaluation metric**, not accuracy.
- Use classical ML algorithms (XGBoost, Random Forest, Gradient Boosting, etc.)
- Must be interpretable — city planners need to understand WHY an intersection is flagged as high-risk
- **Minimum Benchmark:** Accuracy > 70%, weighted F1 > 0.55
- **Stretch Goal:** Accuracy > 80%, weighted F1 > 0.70
- **Required:** SHAP or feature importance analysis — which factors most predict severe accidents?

In [ ]:

################################# Feature Importance Analysis and Plot #########################################
#print(model.feature_importances_)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# get importances
importances = model.feature_importances_

# if X_train is a DataFrame, use its column names
feat_imp = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importances
})

# keep top 30
top_n = 30
feat_imp = feat_imp.sort_values("importance", ascending=False).head(top_n)

feat_imp1 = feat_imp.sort_values("importance", ascending=True).head(top_n)

print("\nTop 30 most important features:")
for i, row in feat_imp1.tail(30).iloc[::-1].iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")


# plot
plt.figure(figsize=(10, 8))
plt.barh(feat_imp["feature"][::-1], feat_imp["importance"][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(f"Top {top_n} Feature Importances")
plt.tight_layout()
plt.show()

In [ ]:
import shap

# create explainer (TreeExplainer is optimized for XGBoost)
explainer = shap.TreeExplainer(model)

# compute SHAP values (use a sample for speed if needed)
X_sample = X_train.sample(100000, random_state=42)

shap_values = explainer.shap_values(X_sample)

In [ ]:
# Top drivers for all classes
for i in range(4):
    print('*****************************')
    print(f"Traffic Severity Level: {i+1}")
    
    shap.summary_plot(shap_values[:,:,i], X_sample)

In [ ]:
# Top drivers for the most severe accidents (level 4) in clear bar chart
shap.summary_plot(shap_values[:,:,3], X_sample, plot_type="bar")

In [ ]:

# Define path relative to notebook
model_path = "../models/model1_traditional_ml/EW_xgb_model.pkl"

# Create folder if not exists
os.makedirs(os.path.dirname(model_path), exist_ok=True)

# Save model
joblib.dump(model, model_path)

print(f"Model saved to {model_path}")